In [4]:
# [목적] 작업별 입력과 답변 예시를 준비합니다.
# 예시들은 새 요청과 가장 비슷한 사례를 찾는 기준이 되며, 이후 프롬프트에 참고 자료로 들어갑니다.
examples = [
    {
        "instruction": "당신은 회의록 작성 전문가 입니다....",
        "input": "2023년 12월 25일, XYZ 회사의 마케팅 전략 회의가 오후 3시에 시작되었다...",
        "answer": """..."""
    },
    {
        "instruction": "당신은 요약 전문가 입니다. 다음 주어진 정보를 바탕으로 내용을 요약해 주세요",
        "input": "이 문서는 '지속 가능한 도시 개발을 위한 전략'에 대한 20페이지 분량의...",
        "answer": """문서 요약: 지속 가능한 도시 개발을 위한 전략 보고서..."""
    },
{
    "instruction": "당신은 문장 교정 전문가 입니다. 다음 주어진 문장을 교정해 주세요",
    "input": "우리 회사는 새로운 마케팅 전략을 도입하려고 한다....",
    "answer": "본 회사는 새로운 마케팅 전략을 도입함으로써, ..."
},
]

In [5]:
from langchain_teddynote.prompts import CustomExampleSelector
from langchain_openai import OpenAIEmbeddings

# 커스텀 예제 선택기 생성
custom_selector = CustomExampleSelector(examples, OpenAIEmbeddings())

# 커스텀 예제 선택기를 사용했을 때 결과
custom_selector.select_examples({"instruction": "다음 문장으로 회의록을 작성해 주세요"})

[{'instruction': '당신은 문장 교정 전문가 입니다. 다음 주어진 문장을 교정해 주세요',
  'input': '우리 회사는 새로운 마케팅 전략을 도입하려고 한다....',
  'answer': '본 회사는 새로운 마케팅 전략을 도입함으로써, ...'}]

In [6]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{instruction}:\n{input}"),
        ("ai", "{answer}"),
    ]
)

custom_fewshot_prompt = FewShotChatMessagePromptTemplate(
    example_selector=custom_selector,  # 커스텀 예제 선택기 사용
    example_prompt=example_prompt,  # 예제 프롬프트 사용
)

custom_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant.",
        ),
        custom_fewshot_prompt,
        ("human", "{instruction}\n{input}"),
    ]
)

In [8]:
from langchain_openai import ChatOpenAI
from langchain_teddynote.messages import stream_response

llm = ChatOpenAI()

chain = custom_prompt | llm  # 체인을 생성

question = {
    "instruction": "회의록을 작성해 주세요",
    "input": "2023년 12월 26일, ABC 기술 회사의 제품 개발 팀은 새로운 모바일 애플리케이션 프로젝트에 대한 주간 진행 상황 회의를 가졌다....",
}

stream_response(chain.stream(question))  # 실행 및 결과 출력

저는 실제 회의록을 만들 수 없습니다. 하지만 회의록을 작성하는 방법을 안내해 드릴 수 있습니다. 

1. 회의록 제목: ABC 기술 회사 제품 개발 팀 주간 진행 상황 회의록
2. 날짜 및 시간: 2023년 12월 26일, 오전 10시
3. 참석자: 팀장, 프로젝트 매니저, 개발자 등
4. 회의 안건: 새로운 모바일 애플리케이션 프로젝트의 진행 상황과 이슈 소개
5. 주요 회의 내용 요약:
   - 이번 주 프로젝트에서 발생한 중요한 이슈들에 대해 토의
   - 각 팀원이 수행한 작업 및 진행 상황 공유
   - 다음 단계 계획 및 일정 조정 사항에 대한 협의
   - 향후 업무 분담 및 업무 우선순위에 대한 논의

이렇게 회의록을 작성하면 후속 조치 및 의사 결정이 쉽게 이루어지도록 도움이 될 것입니다.